In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import pywt

In [4]:
import torch
from torch import nn
import numpy as np
from torch.utils.data import DataLoader
import pywt
import torchvision
from torchvision import transforms

In [7]:
from src.architechures.cnn import Simple_CNN_1D
from src.architechures.transformers import TS_Transformer
from src.architechures.resnet2d import ResNet
from src.trainers import SIM_CLR_Trainer
from src.metrics import get_scores, metrics
from src.datasets import DronesDataset, WiFiDataset
from matplotlib import pyplot as plt

In [8]:
class Mlp(nn.Module):
    def __init__(self, in_features, out_features, apply_softmax = False):
        super(Mlp, self).__init__()
        self.ln1 = nn.Linear(in_features = in_features, out_features = in_features)
        self.relu = nn.ReLU()
        self.ln2 = nn.Linear(in_features = in_features, out_features = out_features)
        self.apply_softmax = apply_softmax
        self.softmax = nn.Softmax()
    def forward(self, x):
        out = self.ln2(self.relu(self.ln1(x)))
        if self.apply_softmax:
            out = self.softmax(out)
        return out

class Amplifier(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Amplifier, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length
        self.noise_std = noise_std

    def forward(self, x):
        scale = torch.randn(x.shape, device = x.device) * self.noise_std + 1

        x *= scale

        return (x)

class Bias(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Bias, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length
        self.noise_std = noise_std

    def forward(self, x):
        bias = torch.randn(1, device = x.device) * self.noise_std

        x += bias

        return (x)

class Noise(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Noise, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length
        self.noise_std = noise_std

    def forward(self, x):
        noise = torch.randn(x.shape, device = x.device) * self.noise_std

        x += noise

        return (x)

class Inverse(nn.Module):
    
    def __init__(self, in_channels, signal_length, noise_std=0):
        super(Inverse, self).__init__()
        self.in_channels = in_channels
        self.signal_length = signal_length

    def forward(self, x):
        inverse_indices = [x.shape[-1] - 1 - i for i in range(x.shape[-1])]
        
        return x[..., inverse_indices]
    
class Augmentation_Masked(nn.Module):
    def __init__(self, in_channels, singal_length, aug_module, noise_std = 0, num_bits = 10, prob = 0.5):
        
        super(Augmentation_Masked, self).__init__()

        assert singal_length % num_bits == 0

        self.size = in_channels * singal_length
        self.noise_std = noise_std
        self.num_bits = num_bits
        self.in_channels = in_channels
        self.singal_length = singal_length
        self.bit_size = singal_length // num_bits
        self.prob = prob


        self.layers = nn.ModuleList(
            [
                aug_module(in_channels, singal_length // num_bits, noise_std)

                for i in range(num_bits)
            ]
        )

    def forward(self, x):
        new_x = torch.zeros(x.shape, device = x.device)
        
        for i in range(self.num_bits):
            
            if np.random.choice([0,1], p = [1-self.prob, self.prob]):
                
                new_x[...,i * self.bit_size: (i+1) * self.bit_size] =\
                self.layers[i](
                    x[...,i * self.bit_size: (i+1) * self.bit_size]
                )
                
            else:
                
                new_x[...,i * self.bit_size: (i+1) * self.bit_size] =\
                    x[...,i * self.bit_size: (i+1) * self.bit_size]
        
        return new_x



In [10]:
class Viewmaker(nn.Module):
    def __init__(self, size, noise_std=0, epsilon = 0.1):
        super(Viewmaker, self).__init__()

        self.size = size
        self.noise_std = noise_std
        self.ln1 = nn.Linear(self.size, self.size)
        self.relu = nn.ReLU()
        self.ln2 = nn.Linear(self.size, self.size)
        self.bn = nn.BatchNorm1d(self.size)
        self.epsiolon = epsilon

    def forward(self, x):
        inp = x
        x = x.reshape(x.shape[0], self.size)
        
        x = (x + torch.randn(x.shape, device = x.device) * self.noise_std) / (1+self.noise_std)
        
        x = self.ln2(self.relu(self.ln1(x)))
        
        x = x.reshape(inp.shape)
        
        x = self.epsiolon * x / torch.norm(x)
        
        return (x + inp)
         

In [22]:
def train_epochs(model, mlp_instance, mpl_cluster, augs, train_loader, test_loader, targets, optimizer, num_epochs=100, report_lag = 5):
    for epoch in range(num_epochs):
        loss = trainer.train_epoch(
            model, 
            mlp_instance, 
            mpl_cluster,
            augs,
            train_loader, 
            optimizer,
            'cuda',
            epoch_type='augs' if epoch % 2 else 'features extractor',
        )

        print(f"Epoch : {epoch}, loss: {loss}")
        
        if epoch % report_lag == 0:
            
            train_f, test_f =\
            trainer.get_features(model, train_loader, 'cuda',  None),\
            trainer.get_features(model, test_loader, 'cuda',  None)
            
            scores = get_scores(
                train_f, test_f, 
                features_type='features', clusters_num=40, criterion = 'right-sided')
            
            evals = metrics(1 - np.array(scores), targets, treshold=0.95)
            
            print(epoch, evals)

In [14]:
def scale(array, mean = 1.5, std = 7239):
    return (array - mean) / std


def normalize_energy(tensor):
    if not torch.is_tensor(tensor):
        tensor = torch.tensor(tensor)
    
    energy = torch.sqrt((tensor**2).sum(0)).mean()
    
    return tensor / energy


def wavlet_transform(x, scales):
    coefs,_ =\
    pywt.cwt(1j*x[0] + x[1], np.arange(1, scales + 1), "cmor1.5-1.0")
    return torch.tensor(np.abs(coefs)).reshape(1, scales, x.shape[1])


dataset_train = DronesDataset('data_drones_500.h5', uavs=[1,2,3], bursts=[1],
                              transform=transforms.Compose([
                                    scale,
                                    lambda x: wavlet_transform(x, 256),
                                    torchvision.transforms.Normalize(0.5, 0.3),
                                    torchvision.transforms.Resize(size = (120,120))]
                                )
                             )

dataset_test = DronesDataset('data_drones_500.h5', uavs=[1,2,3,4], bursts=[1],
                              transform=transforms.Compose([
                                    scale,
                                    lambda x: wavlet_transform(x, 256),
                                    torchvision.transforms.Normalize(0.5, 0.3),
                                    torchvision.transforms.Resize(size = (120,120))]
                                )
                            )

'''

dataset_train = WiFiDataset('data_multi_labels_1024_test_6.h5', uavs=[1,2,3,4], #bursts=[1],
                              transform=lambda x: normalize_energy(scale(x)))

dataset_test = WiFiDataset('data_multi_labels_1024_test_6.h5', uavs=[1,2,3,4,5], #bursts=[1],
                             transform=lambda x: normalize_energy(scale(x)) )
'''

"\n\ndataset_train = WiFiDataset('data_multi_labels_1024_test_6.h5', uavs=[1,2,3,4], #bursts=[1],\n                              transform=lambda x: normalize_energy(scale(x)))\n\ndataset_test = WiFiDataset('data_multi_labels_1024_test_6.h5', uavs=[1,2,3,4,5], #bursts=[1],\n                             transform=lambda x: normalize_energy(scale(x)) )\n"

In [15]:
len(dataset_test)

26883

In [16]:
train_loader = DataLoader(dataset_train, batch_size = 128, num_workers = 10, shuffle=True)

test_loader = DataLoader(dataset_test, batch_size = 128, num_workers = 10, shuffle=False)

In [25]:
test_targets = dataset_test.drone_ids > 3

In [ ]:
lr_curves = [[]]
for num_bits in [12]:
    
    augs = nn.ModuleList(
        [
        Viewmaker(size = 120*120, noise_std=0.5, epsilon=1),
        Viewmaker(size = 120*120, noise_std=0.5, epsilon=1),
        ]
    )
    '''
    augs = nn.ModuleList(
        [Augmentation_Masked(2, 120, Noise,     num_bits = num_bits, prob=0.8, noise_std=0.1),
         Augmentation_Masked(2, 120, Bias,      num_bits = num_bits, prob=0.8, noise_std=0.1),
         Augmentation_Masked(2, 120, Inverse,   num_bits = num_bits, prob=0.8, noise_std=0.1),
         Augmentation_Masked(2, 120, Amplifier, num_bits = num_bits, prob=0.8, noise_std=0.1),]
    )
    '''
    
    trainer = SIM_CLR_Trainer(augs)

    model = ResNet(layers = [1]*4, num_classes=1)
    mlp_instance = Mlp(512, 20, False)
    mpl_cluster = Mlp(512, 20, True)

    optimizer = torch.optim.Adam([
        {'params': model.parameters(), 'lr': 1e-4}, 
        {'params': mlp_instance.parameters(), 'lr': 1e-4}, 
        {'params': mpl_cluster.parameters(), 'lr': 1e-4},
        {'params': augs.parameters(), 'lr': 1e-3}
    ])

    lr_curves[-1].append(
        train_epochs(model, mlp_instance, mpl_cluster, augs, train_loader, test_loader, test_targets, optimizer, num_epochs=60, report_lag = 4)
    )